# Sentence SHAP values

Compute attributions for the maximum category probability or for all nine category outputs. Select the model run by its context window.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
import pandas as pd
import glob
import shap
import torch
from transformers import pipeline, AutoTokenizer
import pickle
import numpy as np
import os
from pathlib import Path
from shared.labels import CATEGORY_INDEX
from shared.shap_values import truncate_long_texts, merge_shap_values_with_offsets

CONTEXT_WINDOW = 4
paths.prepare_directories()
CV_BASE_PATH = str(paths.sentence_run(CONTEXT_WINDOW))
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


## Reference score

In [ ]:
from transformers import AutoTokenizer, BertForSequenceClassification
from scipy.special import expit
import shap
from shap import maskers
import numpy as np
import os
import pickle
import pandas as pd

for fold in range(0, 10):
    print(f'\n Processing fold {fold}...')
    shap_path = os.path.join(CV_BASE_PATH, f'fold_{fold}/logs/shap_data_fold_{fold}.pkl')
    if not os.path.exists(shap_path):
        print(f' Missing SHAP input for fold {fold}: {shap_path}')
        continue
    with open(shap_path, 'rb') as f:
        shap_data = pickle.load(f)
    model_path = paths.resolve_checkpoint(shap_data['model_path'], Path(CV_BASE_PATH) / f'fold_{fold}')
    test_data = shap_data['tokenized_test_data']
    test_texts = test_data['raw_texts']
    test_ids = test_data['letter_ids']
    test_sents = test_data['sentence_ids']
    print(f' Model: {model_path}')
    print(f' Test instances: {len(test_texts)}')
    tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
    model = BertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    test_texts_trunc = truncate_long_texts(test_texts, test_ids, tokenizer)

    def _to_str_list(x):
        if isinstance(x, str):
            return [x]
        if isinstance(x, np.ndarray):
            x = x.tolist()
        if isinstance(x, tuple):
            x = list(x)
        if not isinstance(x, list):
            x = [str(x)]
        return [t if isinstance(t, str) else str(t) for t in x]

    def predict_fn(texts):
        texts = _to_str_list(texts)
        inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(
            device,
        )
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = expit(logits.cpu().numpy())
            return probs.max(axis=1)
    masker = maskers.Text(tokenizer)
    explainer = shap.Explainer(predict_fn, masker)
    shap_values = explainer(test_texts_trunc)
    final_dfs = []
    for i, (text, shap_vals, text_id) in enumerate(zip(test_texts_trunc, shap_values.values, test_ids)):
        merged_tokens, merged_shap = merge_shap_values_with_offsets(text, shap_vals, tokenizer)
        shap_df = pd.DataFrame(
            {
                'token': merged_tokens,
                'shap_Ref': merged_shap,
                'text_id': text_id,
                'sent_id': test_sents[i],
            },
        )
        final_dfs.append(shap_df)
    merged_shap_df = pd.concat(final_dfs, ignore_index=True)
    print('\n--- Final Merged Tokens & SHAP Values ---')
    print('[LETTER TEXT]')
    output_path = os.path.join(CV_BASE_PATH, f'fold_{fold}/logs/shap_output_fold_{fold}.csv')
    merged_shap_df.to_csv(output_path, index=False)
    print(f' SHAP results for fold {fold} saved to: {output_path}')


## Combined reference-score table

In [ ]:
import pandas as pd
import glob
import os

base_dir = CV_BASE_PATH
output_dir = os.path.join(base_dir, 'general')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'shap_output_all_folds.csv')
fold_paths = glob.glob(os.path.join(base_dir, 'fold_*/logs/shap_output_fold_*.csv'))
dfs = []
for path in fold_paths:
    try:
        fold_number = int(path.split('fold_')[1].split('/')[0])
    except Exception:
        print(f' Could not read the fold number from: {path}')
        continue
    print(f'\n Processing fold {fold_number}...')
    shap_df = pd.read_csv(path)
    shap_df['fold'] = fold_number
    pred_path = os.path.join(base_dir, f'fold_{fold_number}', 'predictions.csv')
    if not os.path.exists(pred_path):
        print(f' Missing prediction file for fold {fold_number}: {pred_path}')
        continue
    pred_df = pd.read_csv(pred_path)
    pred_df = pred_df.rename(columns={'letter_id': 'text_id', 'sentence_id': 'sent_id'})
    join_cols = ['text_id', 'sent_id']
    merge_cols = ['true_labels', 'true_names', 'pred_labels', 'pred_names']
    merged = pd.merge(shap_df, pred_df[join_cols + merge_cols], on=join_cols, how='left')
    dfs.append(merged)
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_path, index=False)
    print(f'\n Combined SHAP table with predictions saved to:\n{output_path}')
else:
    print(' No SHAP output files found.')


## Category scores

In [ ]:
from transformers import AutoTokenizer, BertForSequenceClassification
from scipy.special import expit
import shap
from shap import maskers
import numpy as np
import os
import pickle
import pandas as pd

INDEX_TO_CATEGORY = {v: k for k, v in CATEGORY_INDEX.items()}
DEBUG = False
for fold in range(0, 10):
    print(f'\n Processing fold {fold}...')
    shap_path = os.path.join(CV_BASE_PATH, f'fold_{fold}/logs/shap_data_fold_{fold}.pkl')
    if not os.path.exists(shap_path):
        print(f' Missing SHAP input for fold {fold}: {shap_path}')
        continue
    with open(shap_path, 'rb') as f:
        shap_data = pickle.load(f)
    model_path = shap_data['model_path']
    test_data = shap_data['tokenized_test_data']
    test_texts = test_data['raw_texts']
    test_ids = test_data['letter_ids']
    test_sents = test_data['sentence_ids']
    print(f' Model: {model_path}')
    print(f' Test instances: {len(test_texts)}')
    tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
    model = BertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    test_texts_trunc = truncate_long_texts(test_texts, test_ids, tokenizer)

    def _to_str_list(x):
        if isinstance(x, str):
            return [x]
        if isinstance(x, np.ndarray):
            x = x.tolist()
        if isinstance(x, tuple):
            x = list(x)
        if not isinstance(x, list):
            x = [str(x)]
        return [t if isinstance(t, str) else str(t) for t in x]

    def predict_fn(texts):
        texts = _to_str_list(texts)
        inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(
            device,
        )
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = expit(logits.cpu().numpy())
            return probs
    masker = maskers.Text(tokenizer)
    explainer = shap.Explainer(predict_fn, masker)
    num_labels = int(model.config.num_labels)
    shap_values = explainer(test_texts_trunc, outputs=list(range(num_labels)))
    final_dfs = []
    for i, (text, text_id) in enumerate(zip(test_texts_trunc, test_ids)):
        sv_i = shap_values.values[i]
        merged_tokens, merged_shap = merge_shap_values_with_offsets(text, sv_i, tokenizer)
        merged_shap = np.asarray(merged_shap)
        if merged_shap.ndim == 1:
            merged_shap = merged_shap[:, None]
        num_classes = merged_shap.shape[1]
        shap_columns = {f'shap_{INDEX_TO_CATEGORY[k]}': merged_shap[:, k] for k in range(num_classes)}
        shap_df = pd.DataFrame(
            {'token': merged_tokens, 'text_id': text_id, 'sent_id': test_sents[i], **shap_columns},
        )
        final_dfs.append(shap_df)
    merged_shap_df = pd.concat(final_dfs, ignore_index=True)
    print('\n--- Final Merged Tokens & SHAP Values ---')
    print('[LETTER TEXT]')
    output_path = os.path.join(CV_BASE_PATH, f'fold_{fold}/logs/shap_output_multilabel_fold_{fold}.csv')
    merged_shap_df.to_csv(output_path, index=False)
    print(f' SHAP results for fold {fold} saved to: {output_path}')


## Combined category-score table

In [ ]:
import pandas as pd
import glob
import os

base_dir = CV_BASE_PATH
output_dir = os.path.join(base_dir, 'general')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'shap_output_multilabel_all_folds.csv')
fold_paths = glob.glob(os.path.join(base_dir, 'fold_*/logs/shap_output_multilabel_fold_*.csv'))
dfs = []
for path in fold_paths:
    try:
        fold_number = int(path.split('fold_')[1].split('/')[0])
    except Exception:
        print(f' Could not read the fold number from: {path}')
        continue
    print(f'\n Processing fold {fold_number}...')
    shap_df = pd.read_csv(path)
    shap_df['fold'] = fold_number
    pred_path = os.path.join(base_dir, f'fold_{fold_number}', 'predictions.csv')
    if not os.path.exists(pred_path):
        print(f' Missing prediction file for fold {fold_number}: {pred_path}')
        continue
    pred_df = pd.read_csv(pred_path)
    pred_df = pred_df.rename(columns={'letter_id': 'text_id', 'sentence_id': 'sent_id'})
    join_cols = ['text_id', 'sent_id']
    merge_cols = ['true_labels', 'true_names', 'pred_labels', 'pred_names']
    merged = pd.merge(shap_df, pred_df[join_cols + merge_cols], on=join_cols, how='left')
    dfs.append(merged)
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_path, index=False)
    print(f'\n Combined SHAP table with predictions saved to:\n{output_path}')
else:
    print(' No SHAP output files found.')
